# ST-AWFD CNN autoencoder

ST-AWFD D2의 Step 1에서 정상 lot의 파형을 복원하는 1D CNN denoising autoencoder를 학습한다. 정상 reference로 정한 경보 기준과 공개 mixed holdout의 오경보·이상 탐지 결과를 비교한다. 학습 cell은 모델과 결과를 새로 저장한다.


In [ ]:
from pathlib import Path  # 이 노트북이 위치한 PE 실습 폴더를 찾는다.
import json  # 실제 run manifest와 split 요약을 읽는다.
import sys  # 같은 폴더의 검증된 training/inference 모듈을 import 경로에 추가한다.
import pandas as pd  # source와 result 표를 확인하고 엑셀 산출을 검사한다.
import matplotlib.pyplot as plt  # 실제 epoch loss와 운영 drift를 시각화한다.
from IPython.display import display  # 표/요약을 노트북 output에 읽기 좋게 표시한다.

PROJECT_DIR = Path.cwd().resolve()  # Jupyter kernel을 프로젝트 폴더에서 실행한다는 가정을 절대경로로 고정한다.
if not (PROJECT_DIR / "st_awfd_pipeline.py").exists():  # Jupyter가 상위 폴더에서 시작해 상대경로 import가 실패하는 경우를 감지한다.
    raise FileNotFoundError("프로젝트 폴더에서 Jupyter를 시작하세요.")  # workspace root에서 열었을 때 project 폴더로 이동한다.
sys.path.insert(0, str(PROJECT_DIR))  # 재사용할 공식 pipeline의 module path를 우선 탐색하도록 한다.
from st_awfd_pipeline import DATA_DIR, MODEL_DIR, RESULTS_DIR, read_d2_source, run_experiment, score_new_csv  # 데이터검증·GPU학습·추론·저장 경로를 한 구현에서 공유한다.

print(f"Project: {PROJECT_DIR}")  # 실행할 프로젝트 경로를 눈으로 확인한다.
print(f"D2 archive exists: {(DATA_DIR / 'D2.zip').exists()}")  # source archive 누락으로 잘못 실행하는 일을 사전에 확인한다.

## 공개 데이터와 split

D2의 실제 행·lot·센서 개수와 공식 train/test 구성을 읽는다. MaterialID는 생산 lot이고, duration_ms는 정규화된 상대 시간이다. 익명 센서값은 원본에서 z-score 처리돼 있다. 실행 요약에 원본 archive의 SHA-256과 실제 parse 결과를 기록한다.


In [ ]:
source = read_d2_source()  # 공식 원본 압축파일을 읽고 필수 열·결측·ID별 라벨·공식 split 일관성을 검증한다.
feature_columns = [column for column in source.columns if column.startswith("feature_")]  # 모델 입력 후보인 익명 센서 열만 명시적으로 분리한다.
wafer_split = source.groupby(["is_test", "target"])["MaterialID"].nunique().rename("MaterialID_count").reset_index()  # row 수 대신 모델의 group 단위인 MaterialID 수를 계산한다.
print(f"Rows={len(source):,} | IDs={source['MaterialID'].nunique():,} | anonymous channels={len(feature_columns)}")  # 원본 규모와 채널 수를 출력한다.
display(wafer_split)  # normal-only train과 mixed test의 실제 구성을 표시한다.

## CNN 학습과 평가

정상 파형에 노이즈를 더하고 원래 파형을 복원하는 1D CNN을 학습한다. 최대 150 epoch이며 정상 validation MSE를 기준으로 early stopping과 checkpoint를 선택한다. CUDA가 있으면 GPU를 사용하고 그 외에는 CPU로 실행한다.


In [ ]:
def report_epoch(event):  # pipeline이 실제로 끝낸 epoch event를 화면에 기록한다.
    if event["epoch"] == 1 or event["epoch"] % 10 == 0:  # 실행 output이 과도하게 길어지지 않도록 주기적으로 관측값을 출력한다.
        print(f"epoch={event['epoch']:03d} | train denoising MSE={event['train_loss']:.6f} | validation normal MSE={event['validation_loss']:.6f} | best={event['best_epoch']:03d} | {event['device']}")  # elapsed time이 아니라 학습 중 측정한 loss와 실제 device를 표시한다.

run = run_experiment(PROJECT_DIR, calibration_target_fpr=0.05, review_fraction=0.10, progress_callback=report_epoch)  # 정상-only GPU fitting부터 official mixed holdout 추론과 Excel report까지 다시 실행한다.
summary = run["summary"]  # 실제 실행이 저장한 source·장치·split·지표를 한 변수로 받는다.
metrics = summary["experiment"]["primary_holdout_metrics"]  # primary temporal model의 retrospective holdout 결과를 읽는다.
print(f"GPU: {summary['device']['device_name']} | CUDA={summary['device']['cuda_available']} | torch={summary['device']['torch_version']}")  # 실행 머신과 CUDA 사용여부를 실제 runtime에서 증명한다.
print(f"Best epoch={summary['experiment']['best_epoch']} | train-normal={(summary['dataset']['official_train_normal_wafers'] - summary['experiment']['calibration_normal_wafers'])} | calibration-normal={summary['experiment']['calibration_normal_wafers']}")  # split 수와 best checkpoint epoch를 표시한다.
print(f"Official holdout: n={metrics['n_wafers']} | anomaly prevalence={metrics['anomaly_prevalence']:.1%} | AP={metrics['average_precision']:.3f} | ROC-AUC={metrics['roc_auc']:.3f}")  # 높은 ranking 점수를 benchmark prevalence와 함께 제시한다.
print(f"Validation normal alarm={metrics['calibration_normal_false_alarm_rate']:.1%}; holdout-normal FPR={metrics['test_normal_false_alarm_rate']:.1%} (Wilson 95% CI {metrics['test_normal_fpr_wilson_95pct'][0]:.1%}–{metrics['test_normal_fpr_wilson_95pct'][1]:.1%})")  # threshold의 normal distribution 전이 실패를 숨기지 않고 보여준다.
print(f"Holdout alert fraction={metrics['test_alert_fraction']:.1%}; 10% review queue captures {metrics['review_queue_true_positives']} of {metrics['n_anomalous']} anomalies")  # threshold alarms와 fixed-capacity review ranking을 구분해 출력한다.
print(f"Excel: {run['excel_path']}")  # 생성한 다중 시트 판단자료 위치를 사용자에게 알려준다.

## Epoch별 loss

Train loss는 노이즈 입력의 복원 MSE이고, validation loss는 별도 정상 lot의 noise-free trace 복원 MSE다. 두 값은 train에서 맞춘 표준화 공간에서 계산한다. 정상 validation loss가 가장 낮은 weight를 저장한다.


In [ ]:
history = run["history"]  # 이번 실행에서 optimizer가 실제 기록한 epoch table을 사용한다.
figure, axis = plt.subplots(figsize=(10, 4.5))  # train/validation 곡선을 읽을 수 있는 크기의 plot을 만든다.
axis.plot(history["epoch"], history["train_denoising_mse"], label="Train denoising MSE", color="#255c85")  # 매 epoch의 실제 train objective를 그린다.
axis.plot(history["epoch"], history["validation_normal_mse"], label="Validation normal MSE", color="#d87927")  # best epoch 선택에 사용한 normal validation score를 그린다.
axis.axvline(summary["experiment"]["best_epoch"], linestyle="--", color="#303030", label=f"Best epoch {summary['experiment']['best_epoch']}")  # validation loss가 가장 낮은 저장 checkpoint 위치를 표시한다.
axis.set_xlabel("Epoch")  # 가로축이 실제 optimizer 학습 회수임을 명시한다.
axis.set_ylabel("MSE in train-fitted z space")  # 세로축이 원래 물리단위가 아님을 명시한다.
axis.set_title("ST-AWFD D2 Step 1 | normal-only temporal autoencoder")  # 실험 데이터와 학습 구조를 제목에 적는다.
axis.grid(alpha=0.2)  # 작은 차이도 읽을 수 있게 약한 grid를 둔다.
axis.legend(frameon=False)  # 두 loss 및 best epoch 의미를 설명한다.
figure.tight_layout()  # title/axis label이 figure 밖으로 잘리지 않게 한다.
plt.show()  # 실행 중 생성한 학습 곡선을 notebook에 직접 표시한다.
display(history.tail(10))  # 최신 epoch의 실제 숫자 기록을 함께 보여준다.

## 순위와 경보 기준

AP와 ROC-AUC는 이상 점수의 순위 지표다. 정상 calibration 점수의 quantile로 정한 threshold를 고정하고, holdout에서 정상 오경보·이상 탐지·검토 큐를 함께 평가한다. 공개 holdout의 이상 비율은 양산 발생률과 구분한다.


In [ ]:
comparison = run["model_comparison"]  # 통계 기준선과 실제 GPU temporal model의 동일 holdout 결과를 가져온다.
columns = ["model", "holdout_average_precision", "holdout_roc_auc", "holdout_test_normal_fpr", "holdout_alert_fraction", "threshold_false_negative"]  # score ranking, normal 경보, anomaly miss를 함께 비교한다.
display(comparison[columns].sort_values("holdout_test_normal_fpr"))  # 한 지표만 최적화하지 않도록 운용 오경보 순으로 보여준다.
drift = run["calibration_audit"]  # validation normal과 independent official test normal의 score 분포를 비교한다.
display(drift[["model", "validation_normal_score_q95", "validation_threshold", "official_test_normal_score_q95", "official_test_normal_exceedance_rate"]])  # 고정 limit과 score distribution shift의 크기를 노출한다.

## 무라벨 CSV 추론

공개 holdout의 20개 lot에서 target과 is_test를 제거해 저장 모델의 CSV 추론을 실행한다. 학습 weight·scaler·threshold를 바꾸지 않고 결과를 검토 큐로 저장한다. 이 예제는 같은 공개 데이터의 API 실행 확인이다.


In [ ]:
demo_ids = source.loc[source["is_test"].eq(1), "MaterialID"].drop_duplicates().head(20)  # 공개 mixed holdout에서 예시 batch ID만 선택한다.
demo_batch = source.loc[source["MaterialID"].isin(demo_ids) & source["StepID"].eq(1)].copy()  # 모델에 필요한 Step 1 trace rows만 구성한다.
demo_batch = demo_batch.drop(columns=["target", "is_test"], errors="ignore")  # 예측 함수로 가는 입력에서 정답과 split 정보를 실제로 제거한다.
demo_input_path = RESULTS_DIR / "st_awfd_label_free_demo_input.csv"  # label 없는 예시 입력을 결과 폴더에 저장한다.
demo_batch.to_csv(demo_input_path, index=False, encoding="utf-8-sig")  # Excel에서 열 수 있고 score_new_csv가 읽을 CSV 파일을 만든다.
demo_queue = score_new_csv(demo_input_path, review_fraction=0.10)  # 저장한 checkpoint와 calibration threshold만으로 실제 inference를 실행한다.
assert "target" not in demo_batch.columns and "is_test" not in demo_batch.columns  # label/split flag가 제거됐다는 구현 조건을 확인한다.
print(f"Unlabeled Step 1 rows={len(demo_batch):,} | unique entities={demo_batch['MaterialID'].nunique()} | score queue={len(demo_queue)}")  # 실제 무라벨 inference의 입력 크기와 결과 수를 출력한다.
display(demo_queue.head(10))  # 정상 reference 초과 여부와 상대 검토 rank를 사람용 결과로 보여준다.
print(f"Queue CSV: {RESULTS_DIR / 'st_awfd_new_batch_demo_queue.csv'}")  # downloadable batch result 위치를 알려준다.

In [ ]:
demo_queue.to_csv(RESULTS_DIR / "st_awfd_new_batch_demo_queue.csv", index=False, encoding="utf-8-sig")  # label-free inference queue를 Excel에서 바로 볼 수 있게 export한다.
print("Excel:", run["excel_path"] or "CSV only")  # 학습이력·모델비교·queue·사후감사 통합 workbook을 안내한다.
print(f"Saved checkpoint: {MODEL_DIR / 'st_awfd_step1_temporal_autoencoder.pt'}")  # 새 무라벨 CSV 재추론에 쓰는 weights/scaler/threshold checkpoint를 표시한다.